# Kaggle × Colab runner

Set `KAGGLE_API_TOKEN` (and `GITHUB_TOKEN` for a private repo) in 🔑 Secrets, edit the options, then run all.

In [ ]:
# @title Options
COMPETITION = "titanic"  # @param {type:"string"}
EXP = "exp001"  # @param {type:"string"}
ARGS = ""  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
USE_DRIVE = True  # @param {type:"boolean"}
# folder in My Drive with one folder per competition: <DRIVE_DIR>/<COMPETITION>/<EXP>/
DRIVE_DIR = "Colab Notebooks/Kaggle"  # @param {type:"string"}
DOWNLOAD = True  # @param {type:"boolean"}
SUBMIT = False  # @param {type:"boolean"}
# release the GPU when the run ends (success or failure) so it stops using compute units
DISCONNECT_WHEN_DONE = True  # @param {type:"boolean"}


In [ ]:
# @title Setup
import importlib
import os
import sys

from google.colab import userdata

REPO = "Moriyuki-S/kaggle_ws"
REPO_DIR = "/content/kaggle_ws"
DATA_DIR = f"/content/data/{COMPETITION}"

for k in ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY", "GITHUB_TOKEN"]:
    try:
        os.environ[k] = userdata.get(k)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        pass

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DRIVE_ROOT"] = DRIVE_DIR  # read by kaggle_ws.get_paths

token = os.environ.get("GITHUB_TOKEN")
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
if not os.path.exists(REPO_DIR):
    !git clone -q {url} {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH} && git log -1 --oneline

!pip install -q -U kaggle && pip install -q --no-deps -e .

# the editable install only reaches new processes; /content/kaggle_ws would otherwise shadow the package
sys.path.insert(0, f"{REPO_DIR}/src")
sys.modules.pop("kaggle_ws", None)
importlib.invalidate_caches()

# a competition can opt out of the full download with a .skip_download file
if DOWNLOAD and not os.path.exists(DATA_DIR) and not os.path.exists(f"competitions/{COMPETITION}/.skip_download"):
    !bash scripts/download.sh {COMPETITION} {DATA_DIR}

In [ ]:
# @title Run
# stopping the cell by hand keeps the runtime, so you can rerun without setting up again
stopped = False
try:
    import shutil
    from datetime import datetime
    from zoneinfo import ZoneInfo

    from kaggle_ws import get_paths

    output = get_paths(COMPETITION).output
    print("outputs:", output)
    sub = output / EXP / "submission.csv"
    sub.unlink(missing_ok=True)  # never reuse a stale submission from a previous run

    !python competitions/{COMPETITION}/exp/{EXP}/train.py {ARGS}
    exit_code = _exit_code  # noqa: F821  (set by IPython after a ! command)
    stopped = exit_code in (-2, 130)  # SIGINT from the stop button
    if exit_code != 0:
        raise RuntimeError(f"train.py failed (exit code {exit_code})")
    has_submission = sub.exists()
    if SUBMIT and not has_submission:
        raise FileNotFoundError(f"train.py did not write {sub}")
    if not has_submission:
        print(f"no submission written: {sub}")

    if USE_DRIVE and has_submission:
        ts = datetime.now(ZoneInfo("Asia/Tokyo")).strftime("%Y%m%d_%H%M%S")
        dst = f"{output}/submissions/{EXP}_{ts}.csv"
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        sub = shutil.copy(sub, dst)
        print("saved:", sub)

    if SUBMIT:
        !kaggle competitions submit -c {COMPETITION} -f "{sub}" -m "{EXP} {ARGS}"
except KeyboardInterrupt:
    stopped = True
    raise
finally:
    if DISCONNECT_WHEN_DONE and not stopped:
        from google.colab import runtime

        print("disconnecting the runtime; outputs are on Drive" if USE_DRIVE else "disconnecting the runtime")
        runtime.unassign()
